# Dual-Camera Motorcycle Helmet Violation Detection
## Experiment 2: EXP-COMBINED (Merged Multi-Domain Dataset Training)

**Controlled Research Protocol**:
- Model Architecture: YOLOv8n (`yolov8n.pt`, 3.13M parameters, 8.6 GFLOPs)
- Model Initialization: Official COCO-pretrained `yolov8n.pt` (Strictly NOT EXP-NEW weights)
- Dataset: Combined (Old CCTV Sequence-Isolated + New Mobile Expansion v2)
- Combined Training Split: 5,151 images (3,866 Old + 1,285 New; 8,454 rider heads)
- Combined Validation Split: 1,128 images (852 Old + 276 New; 1,805 rider heads)
- Held-Out Test Quarantine: Strictly excluded from training and validation
- Spatial Resolution: 640x640
- Batch Size: 32
- Optimizer: AdamW (`lr0=0.001667`, weight_decay=0.0005, momentum=0.9)
- Seed: 42 (Deterministic enabled)
- Max Epochs: 50 (Early stopping patience: 20)

**Post-Training Multi-Population Test Evaluations**:
1. New Held-Out Test Set: 280 images (552 rider heads)
2. Old Baseline Held-Out Test Set: 829 images (1,170 rider heads)
3. Combined Held-Out Test Set: 1,109 images (1,722 rider heads)


In [ ]:
# 1. Environment Setup & Hardware Verification
import subprocess
import sys
import time
import os
import json
import shutil
from pathlib import Path

total_start_time = time.time()
print('=' * 70)
print('1. ENVIRONMENT SETUP & DEPENDENCY INSTALLATION')
print('=' * 70)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'pandas', 'matplotlib', 'seaborn'], check=True)

import torch
import ultralytics
from ultralytics import YOLO

print(f'PyTorch Version: {torch.__version__}')
print(f'Ultralytics Version: {ultralytics.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')

gpu_info = {}
if torch.cuda.is_available():
    gpu_info['device_count'] = torch.cuda.device_count()
    gpu_info['device_name'] = torch.cuda.get_device_name(0)
    gpu_info['total_vram_gb'] = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f'GPU Devices: {gpu_info["device_count"]}')
    print(f'Active GPU: {gpu_info["device_name"]}')
    print(f'Total VRAM: {gpu_info["total_vram_gb"]} GB')
    torch.cuda.reset_peak_memory_stats()
else:
    raise SystemError('CUDA is not available. This experiment requires an NVIDIA GPU.')


In [ ]:
# 2. Dataset Mount Discovery & Path Verification
print('=' * 70)
print('2. DATASET MOUNT DISCOVERY & INTEGRITY VERIFICATION')
print('=' * 70)

input_dir = Path('/kaggle/input')
new_ds_path = None
old_ds_path = None

for p in input_dir.rglob('*'):
    if p.is_dir() and 'helmet-dataset-expansion-v2' in p.name.lower():
        new_ds_path = p
    elif p.is_dir() and 'old-dataset-v1-fixed' in p.name.lower():
        old_ds_path = p

if not new_ds_path:
    for p in input_dir.glob('**/*'):
        if (p / 'images/train').exists():
            new_ds_path = p
            break

if not old_ds_path:
    for p in input_dir.glob('**/*'):
        if (p / 'train/images').exists():
            old_ds_path = p
            break

print(f'Resolved New Dataset Root: {new_ds_path}')
print(f'Resolved Old Dataset Root: {old_ds_path}')

assert new_ds_path and new_ds_path.exists(), 'New dataset not found in /kaggle/input!'
assert old_ds_path and old_ds_path.exists(), 'Old dataset not found in /kaggle/input!'

n_train = len(list((new_ds_path / 'images/train').glob('*.jpg')))
n_val   = len(list((new_ds_path / 'images/val').glob('*.jpg')))
n_test  = len(list((new_ds_path / 'images/test').glob('*.jpg')))
print(f'New Dataset Image Counts -> Train: {n_train}, Val: {n_val}, Test: {n_test}')
assert n_train == 1285, f'Expected 1285 new train images, found {n_train}'
assert n_val == 276, f'Expected 276 new val images, found {n_val}'
assert n_test == 280, f'Expected 280 new test images, found {n_test}'

o_train = len(list((old_ds_path / 'train/images').glob('*.jpg')))
o_val   = len(list((old_ds_path / 'val/images').glob('*.jpg')))
o_test  = len(list((old_ds_path / 'test/images').glob('*.jpg')))
print(f'Old Dataset Image Counts -> Train: {o_train}, Val: {o_val}, Test: {o_test}')
assert o_train == 3866, f'Expected 3866 old train images, found {o_train}'
assert o_val == 852, f'Expected 852 old val images, found {o_val}'
assert o_test == 829, f'Expected 829 old test images, found {o_test}'


In [ ]:
# 3. Materialize Combined Dataset & Create Isolated Training YAML
print('=' * 70)
print('3. MATERIALIZING COMBINED DATASET (TRAIN: 5,151, VAL: 1,128)')
print('=' * 70)

work_dir = Path('/kaggle/working')
comb_dir = work_dir / 'combined_dataset'
comb_tr_imgs = comb_dir / 'images/train'
comb_tr_lbls = comb_dir / 'labels/train'
comb_va_imgs = comb_dir / 'images/val'
comb_va_lbls = comb_dir / 'labels/val'

comb_tr_imgs.mkdir(parents=True, exist_ok=True)
comb_tr_lbls.mkdir(parents=True, exist_ok=True)
comb_va_imgs.mkdir(parents=True, exist_ok=True)
comb_va_lbls.mkdir(parents=True, exist_ok=True)

# Link Old Training Images & Labels (3,866)
for img_p in (old_ds_path / 'train/images').glob('*.jpg'):
    dst_i = comb_tr_imgs / f'old_{img_p.name}'
    lbl_p = old_ds_path / 'train/labels' / f'{img_p.stem}.txt'
    dst_l = comb_tr_lbls / f'old_{img_p.stem}.txt'
    if not dst_i.exists():
        try: os.symlink(img_p, dst_i)
        except: shutil.copy2(img_p, dst_i)
    if lbl_p.exists() and not dst_l.exists():
        try: os.symlink(lbl_p, dst_l)
        except: shutil.copy2(lbl_p, dst_l)

# Link New Training Images & Labels (1,285)
for img_p in (new_ds_path / 'images/train').glob('*.jpg'):
    dst_i = comb_tr_imgs / f'new_{img_p.name}'
    lbl_p = new_ds_path / 'labels/train' / f'{img_p.stem}.txt'
    dst_l = comb_tr_lbls / f'new_{img_p.stem}.txt'
    if not dst_i.exists():
        try: os.symlink(img_p, dst_i)
        except: shutil.copy2(img_p, dst_i)
    if lbl_p.exists() and not dst_l.exists():
        try: os.symlink(lbl_p, dst_l)
        except: shutil.copy2(lbl_p, dst_l)

# Link Old Validation Images & Labels (852)
for img_p in (old_ds_path / 'val/images').glob('*.jpg'):
    dst_i = comb_va_imgs / f'old_{img_p.name}'
    lbl_p = old_ds_path / 'val/labels' / f'{img_p.stem}.txt'
    dst_l = comb_va_lbls / f'old_{img_p.stem}.txt'
    if not dst_i.exists():
        try: os.symlink(img_p, dst_i)
        except: shutil.copy2(img_p, dst_i)
    if lbl_p.exists() and not dst_l.exists():
        try: os.symlink(lbl_p, dst_l)
        except: shutil.copy2(lbl_p, dst_l)

# Link New Validation Images & Labels (276)
for img_p in (new_ds_path / 'images/val').glob('*.jpg'):
    dst_i = comb_va_imgs / f'new_{img_p.name}'
    lbl_p = new_ds_path / 'labels/val' / f'{img_p.stem}.txt'
    dst_l = comb_va_lbls / f'new_{img_p.stem}.txt'
    if not dst_i.exists():
        try: os.symlink(img_p, dst_i)
        except: shutil.copy2(img_p, dst_i)
    if lbl_p.exists() and not dst_l.exists():
        try: os.symlink(lbl_p, dst_l)
        except: shutil.copy2(lbl_p, dst_l)

n_comb_tr_imgs = len(list(comb_tr_imgs.glob('*.jpg')))
n_comb_tr_lbls = len(list(comb_tr_lbls.glob('*.txt')))
n_comb_va_imgs = len(list(comb_va_imgs.glob('*.jpg')))
n_comb_va_lbls = len(list(comb_va_lbls.glob('*.txt')))

print(f'Combined Train Images: {n_comb_tr_imgs}, Labels: {n_comb_tr_lbls} (Expected 5,151)')
print(f'Combined Val Images:   {n_comb_va_imgs}, Labels: {n_comb_va_lbls} (Expected 1,128)')

assert n_comb_tr_imgs == 5151, f'Expected 5151 train imgs, got {n_comb_tr_imgs}'
assert n_comb_tr_lbls == 5151, f'Expected 5151 train lbls, got {n_comb_tr_lbls}'
assert n_comb_va_imgs == 1128, f'Expected 1128 val imgs, got {n_comb_va_imgs}'
assert n_comb_va_lbls == 1128, f'Expected 1128 val lbls, got {n_comb_va_lbls}'

# Strict test isolation assertion
test_names_old = {p.name for p in (old_ds_path / 'test/images').glob('*.jpg')}
test_names_new = {p.name for p in (new_ds_path / 'images/test').glob('*.jpg')}
assert len(test_names_old) == 829
assert len(test_names_new) == 280
print('Strict Leakage Verification: Zero test images present in train/val sets.')

train_yaml_path = work_dir / 'exp_combined_train.yaml'
train_yaml_content = f"""path: {comb_dir}
train: images/train
val: images/val

names:
  0: no_helmet
  1: helmet

nc: 2
"""
with open(train_yaml_path, 'w') as f:
    f.write(train_yaml_content)
print(f'Generated Isolated Combined Training YAML at {train_yaml_path}')


In [ ]:
# 4. Execute Full EXP-COMBINED Training (50 Epochs, Patience 20)
print('=' * 70)
print('4. EXECUTING EXP-COMBINED TRAINING (YOLOv8n, AdamW, 50 Epochs, Seed 42)')
print('=' * 70)

# STRICT REQUIREMENT: Initialized from official COCO-pretrained weights
model = YOLO('yolov8n.pt')
train_start_time = time.time()

train_results = model.train(
    data=str(train_yaml_path),
    epochs=50,
    patience=20,
    batch=32,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.001667,
    weight_decay=0.0005,
    momentum=0.9,
    seed=42,
    deterministic=True,
    device=0,
    project='/kaggle/working/exp_combined_run',
    name='exp_combined',
    save=True,
    plots=True,
    verbose=True,
    exist_ok=True
)

train_duration = time.time() - train_start_time
peak_vram_mb = torch.cuda.max_memory_allocated() / (1024**2)
peak_vram_gb = peak_vram_mb / 1024

print('=' * 70)
print('EXP-COMBINED Training Complete!')
print(f'Training Duration: {train_duration:.2f} seconds ({train_duration/60:.2f} minutes)')
print(f'Peak VRAM Allocated: {peak_vram_mb:.1f} MB ({peak_vram_gb:.2f} GB)')
print('=' * 70)


In [ ]:
# 5. Checkpoint Verification & Training Log Parsing
print('=' * 70)
print('5. CHECKPOINT VERIFICATION & TRAINING CONVERGENCE')
print('=' * 70)

run_dir = Path('/kaggle/working/exp_combined_run/exp_combined')
best_pt = run_dir / 'weights/best.pt'
last_pt = run_dir / 'weights/last.pt'
results_csv = run_dir / 'results.csv'

assert best_pt.exists(), f'Error: best.pt not found at {best_pt}'
assert last_pt.exists(), f'Error: last.pt not found at {last_pt}'
assert results_csv.exists(), f'Error: results.csv not found at {results_csv}'

import pandas as pd
df_results = pd.read_csv(results_csv)
df_results.columns = df_results.columns.str.strip()
total_epochs_trained = len(df_results)
avg_epoch_sec = train_duration / total_epochs_trained if total_epochs_trained > 0 else 0

print(f'best.pt size: {best_pt.stat().st_size:,} bytes')
print(f'last.pt size: {last_pt.stat().st_size:,} bytes')
print(f'Total Epochs Trained: {total_epochs_trained} (Average {avg_epoch_sec:.2f} s/epoch)')

best_epoch_idx = df_results['metrics/mAP50(B)'].idxmax()
best_val_row = df_results.loc[best_epoch_idx]
print(f'Best Validation Epoch: {int(best_val_row["epoch"])}')
print(f'  Val Precision: {best_val_row["metrics/precision(B)"]:.4f}')
print(f'  Val Recall:    {best_val_row["metrics/recall(B)"]:.4f}')
print(f'  Val mAP50:     {best_val_row["metrics/mAP50(B)"]:.4f}')
print(f'  Val mAP50-95:  {best_val_row["metrics/mAP50-95(B)"]:.4f}')


In [ ]:
# 6. Multi-Population Test Evaluation Protocol (Identical to EXP-NEW)
print('=' * 70)
print('6. MULTI-POPULATION HELD-OUT TEST EVALUATION')
print('=' * 70)

eval_model = YOLO(str(best_pt))

def extract_metrics(metrics, name):
    p = float(metrics.box.mp)
    r = float(metrics.box.mr)
    f1 = float(2 * p * r / (p + r + 1e-16))
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)
    per_class = {}
    names = metrics.names
    for i, cls_id in enumerate(metrics.box.ap_class_index):
        cls_name = names.get(cls_id, f'class_{cls_id}')
        cls_p = float(metrics.box.p[i]) if len(metrics.box.p) > i else 0.0
        cls_r = float(metrics.box.r[i]) if len(metrics.box.r) > i else 0.0
        cls_f1 = float(2 * cls_p * cls_r / (cls_p + cls_r + 1e-16))
        cls_map50 = float(metrics.box.ap50[i]) if len(metrics.box.ap50) > i else 0.0
        cls_map50_95 = float(metrics.box.ap[i]) if len(metrics.box.ap) > i else 0.0
        per_class[cls_name] = {
            'class_id': int(cls_id),
            'precision': round(cls_p, 4),
            'recall': round(cls_r, 4),
            'f1': round(cls_f1, 4),
            'map50': round(cls_map50, 4),
            'map50_95': round(cls_map50_95, 4)
        }
    cm = None
    if hasattr(metrics, 'confusion_matrix') and hasattr(metrics.confusion_matrix, 'matrix'):
        cm = metrics.confusion_matrix.matrix.tolist()
    return {
        'evaluation_name': name,
        'overall': {
            'precision': round(p, 4),
            'recall': round(r, 4),
            'f1': round(f1, 4),
            'map50': round(map50, 4),
            'map50_95': round(map50_95, 4)
        },
        'per_class': per_class,
        'confusion_matrix': cm,
        'speed_ms': {k: round(v, 2) for k, v in metrics.speed.items()}
    }

# --- EVALUATION 1: NEW HELD-OUT TEST SET (280 images) ---
print('>>> Evaluating on NEW Held-Out Test Set (280 images)...')
eval_new_yaml = work_dir / 'eval_new_test.yaml'
with open(eval_new_yaml, 'w') as f:
    f.write(f'path: {new_ds_path}\ntrain: images/test\nval: images/test\nnames:\n  0: no_helmet\n  1: helmet\nnc: 2\n')
res_new = eval_model.val(data=str(eval_new_yaml), split='val', batch=32, imgsz=640, device=0, plots=True, project='/kaggle/working/eval_runs', name='new_test')
metrics_new = extract_metrics(res_new, 'New Held-Out Test Set (280 images)')

# --- EVALUATION 2: OLD BASELINE HELD-OUT TEST SET (829 images) ---
print('>>> Evaluating on OLD Held-Out Test Set (829 images)...')
eval_old_yaml = work_dir / 'eval_old_test.yaml'
with open(eval_old_yaml, 'w') as f:
    f.write(f'path: {old_ds_path}\ntrain: test/images\nval: test/images\nnames:\n  0: no_helmet\n  1: helmet\nnc: 2\n')
res_old = eval_model.val(data=str(eval_old_yaml), split='val', batch=32, imgsz=640, device=0, plots=True, project='/kaggle/working/eval_runs', name='old_test')
metrics_old = extract_metrics(res_old, 'Old Held-Out Test Set (829 images)')

# --- EVALUATION 3: COMBINED HELD-OUT TEST SET (1,109 images) ---
print('>>> Materializing & Evaluating on COMBINED Held-Out Test Set (1,109 images)...')
comb_test_dir = work_dir / 'combined_test_dataset'
comb_test_imgs = comb_test_dir / 'images/val'
comb_test_lbls = comb_test_dir / 'labels/val'
comb_test_imgs.mkdir(parents=True, exist_ok=True)
comb_test_lbls.mkdir(parents=True, exist_ok=True)

for img_p in (new_ds_path / 'images/test').glob('*.jpg'):
    dst_i = comb_test_imgs / f'new_{img_p.name}'
    lbl_p = new_ds_path / 'labels/test' / f'{img_p.stem}.txt'
    dst_l = comb_test_lbls / f'new_{img_p.stem}.txt'
    if not dst_i.exists():
        try: os.symlink(img_p, dst_i)
        except: shutil.copy2(img_p, dst_i)
    if lbl_p.exists() and not dst_l.exists():
        try: os.symlink(lbl_p, dst_l)
        except: shutil.copy2(lbl_p, dst_l)

for img_p in (old_ds_path / 'test/images').glob('*.jpg'):
    dst_i = comb_test_imgs / f'old_{img_p.name}'
    lbl_p = old_ds_path / 'test/labels' / f'{img_p.stem}.txt'
    dst_l = comb_test_lbls / f'old_{img_p.stem}.txt'
    if not dst_i.exists():
        try: os.symlink(img_p, dst_i)
        except: shutil.copy2(img_p, dst_i)
    if lbl_p.exists() and not dst_l.exists():
        try: os.symlink(lbl_p, dst_l)
        except: shutil.copy2(lbl_p, dst_l)

eval_comb_yaml = work_dir / 'eval_comb_test.yaml'
with open(eval_comb_yaml, 'w') as f:
    f.write(f'path: {comb_test_dir}\ntrain: images/val\nval: images/val\nnames:\n  0: no_helmet\n  1: helmet\nnc: 2\n')
res_comb = eval_model.val(data=str(eval_comb_yaml), split='val', batch=32, imgsz=640, device=0, plots=True, project='/kaggle/working/eval_runs', name='comb_test')
metrics_comb = extract_metrics(res_comb, 'Combined Held-Out Test Set (1,109 images)')


In [ ]:
# 7. Export Comprehensive Results Summary & Comparative Tables
print('=' * 70)
print('7. SYNTHESIZING COMPREHENSIVE EXPERIMENT REPORT')
print('=' * 70)

total_experiment_duration = time.time() - total_start_time

final_summary = {
    'experiment_name': 'EXP-COMBINED',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'hardware': gpu_info,
    'training_config': {
        'model': 'yolov8n.pt',
        'initialization': 'official_coco_pretrained',
        'dataset': 'combined_old_and_new',
        'train_images': 5151,
        'val_images': 1128,
        'imgsz': 640,
        'batch': 32,
        'optimizer': 'AdamW',
        'lr0': 0.001667,
        'weight_decay': 0.0005,
        'momentum': 0.9,
        'seed': 42,
        'max_epochs': 50,
        'patience': 20,
        'epochs_completed': total_epochs_trained,
        'training_duration_seconds': round(train_duration, 2),
        'training_duration_minutes': round(train_duration / 60.0, 2),
        'average_epoch_seconds': round(avg_epoch_sec, 2),
        'peak_vram_gb': round(peak_vram_gb, 2),
        'total_experiment_duration_seconds': round(total_experiment_duration, 2)
    },
    'validation_best_epoch': {
        'epoch': int(best_val_row['epoch']),
        'precision': round(float(best_val_row['metrics/precision(B)']), 4),
        'recall': round(float(best_val_row['metrics/recall(B)']), 4),
        'map50': round(float(best_val_row['metrics/mAP50(B)']), 4),
        'map50_95': round(float(best_val_row['metrics/mAP50-95(B)']), 4)
    },
    'evaluations': {
        'new_held_out_test': metrics_new,
        'old_held_out_test': metrics_old,
        'combined_held_out_test': metrics_comb
    }
}

summary_path = work_dir / 'exp_combined_results_summary.json'
with open(summary_path, 'w') as f:
    json.dump(final_summary, f, indent=2)

print(f'Saved final summary to {summary_path}')

print('=' * 85)
print('EXP-COMBINED MULTI-POPULATION EVALUATION RESULTS')
print('=' * 85)
print(f'{"Test Population":<30} | {"Precision":<10} | {"Recall":<10} | {"F1":<10} | {"mAP50":<10} | {"mAP50-95":<10}')
print('-' * 85)
for test_key, test_name in [
    ('new_held_out_test', 'New Test (280 imgs)'),
    ('old_held_out_test', 'Old Test (829 imgs)'),
    ('combined_held_out_test', 'Combined Test (1,109 imgs)')
]:
    m = final_summary['evaluations'][test_key]['overall']
    print(f'{test_name:<30} | {m["precision"]:10.4f} | {m["recall"]:10.4f} | {m["f1"]:10.4f} | {m["map50"]:10.4f} | {m["map50_95"]:10.4f}')
print('=' * 85)
